In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [2]:
!pip -q install kagglehub

In [3]:
import kagglehub

dataset_path = kagglehub.dataset_download(
    "xdxd003/ff-c23",
)

print(dataset_path)

Using Colab cache for faster access to the 'ff-c23' dataset.
/kaggle/input/ff-c23


In [6]:
import os

DATASET_PATH = dataset_path

print(os.listdir(DATASET_PATH))

for root, dirs, files in os.walk(DATASET_PATH):
    print(root, "->", len(files), "files")
    if files:
        print("  sample:", files[:5])
    if root.count(os.sep) > DATASET_PATH.count(os.sep) + 2:
        dirs[:] = []

['FaceForensics++_C23']
/kaggle/input/ff-c23 -> 0 files
/kaggle/input/ff-c23/FaceForensics++_C23 -> 0 files
/kaggle/input/ff-c23/FaceForensics++_C23/Face2Face -> 1000 files
  sample: ['479_706.mp4', '481_469.mp4', '184_205.mp4', '315_322.mp4', '645_688.mp4']
/kaggle/input/ff-c23/FaceForensics++_C23/csv -> 10 files
  sample: ['FaceSwap.csv', 'FaceShifter.csv', 'DeepFakeDetection.csv', 'Face2Face.csv', 'Deepfakes.csv']
/kaggle/input/ff-c23/FaceForensics++_C23/Deepfakes -> 1000 files
  sample: ['479_706.mp4', '481_469.mp4', '184_205.mp4', '315_322.mp4', '645_688.mp4']
/kaggle/input/ff-c23/FaceForensics++_C23/DeepFakeDetection -> 1000 files
  sample: ['02_09__kitchen_pan__9TDCEK1Q.mp4', '02_13__exit_phone_room__CP5HFV3K.mp4', '24_10__walking_down_street_outside_angry__356HPS7R.mp4', '09_14__talking_angry_couch__6TEK3ZX0.mp4', '20_03__hugging_happy__Z6V05FXO.mp4']
/kaggle/input/ff-c23/FaceForensics++_C23/original -> 1000 files
  sample: ['123.mp4', '738.mp4', '479.mp4', '660.mp4', '565.mp4'

In [8]:
import pandas as pd
import os

BASE = os.path.join(dataset_path, "FaceForensics++_C23")

csv_dir = os.path.join(BASE, "csv")

for file in os.listdir(csv_dir):
    path = os.path.join(csv_dir, file)
    df = pd.read_csv(path)

    print("\n" + "="*60)
    print(file)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print(df.head(3))


FaceSwap.csv
Shape: (1000, 8)
Columns: ['Unnamed: 0', 'File Path', 'Label', 'Frame Count', 'Width', 'Height', 'Codec', 'File Size(MB)']
   Unnamed: 0             File Path Label  Frame Count  Width  Height Codec  \
0           0  FaceSwap/000_003.mp4  FAKE          303    640     480  h264   
1           1  FaceSwap/001_870.mp4  FAKE          460   1280     720  h264   
2           2  FaceSwap/002_006.mp4  FAKE          310   1280     720  h264   

   File Size(MB)  
0           0.69  
1           2.77  
2           0.96  

FaceShifter.csv
Shape: (1000, 8)
Columns: ['Unnamed: 0', 'File Path', 'Label', 'Frame Count', 'Width', 'Height', 'Codec', 'File Size(MB)']
   Unnamed: 0                File Path Label  Frame Count  Width  Height  \
0           0  FaceShifter/000_003.mp4  FAKE          396    640     480   
1           1  FaceShifter/001_870.mp4  FAKE          460   1280     720   
2           2  FaceShifter/002_006.mp4  FAKE          693   1280     720   

  Codec  File Size(MB)  


In [11]:
import pandas as pd
import os

BASE = os.path.join(dataset_path, "FaceForensics++_C23")
CSV = os.path.join(BASE, "csv", "FF++_Metadata_Shuffled.csv")

df = pd.read_csv(CSV)

print("Total:", len(df))
print("\nLabels:")
print(df["Label"].value_counts())

print("\nFile paths:")
print(df["File Path"].head(20).to_string(index=False))

Total: 7000

Labels:
Label
FAKE    6000
REAL    1000
Name: count, dtype: int64

File paths:
                                  original/500.mp4
                             Face2Face/944_032.mp4
                             Face2Face/024_073.mp4
DeepFakeDetection/06_18__walking_down_street_ou...
                              FaceSwap/350_349.mp4
                           FaceShifter/424_408.mp4
                                  original/748.mp4
                                  original/215.mp4
                                  original/362.mp4
                        NeuralTextures/589_565.mp4
DeepFakeDetection/03_07__talking_against_wall__...
                                  original/406.mp4
                              FaceSwap/279_298.mp4
                             Face2Face/398_457.mp4
                        NeuralTextures/712_716.mp4
                           FaceShifter/254_261.mp4
                                  original/026.mp4
                             Face2Face/26

In [12]:
# Check how many unique source IDs we can identify

def get_source_id(path):
    name = os.path.basename(path).replace(".mp4", "")

    # original/500.mp4 -> 500
    if "_" not in name:
        return name

    # 873_872.mp4 -> source video is 873
    return name.split("_")[0]

df["source_id"] = df["File Path"].apply(get_source_id)

print("Unique source IDs:", df["source_id"].nunique())

print("\nVideos per source ID:")
print(df.groupby("source_id").size().value_counts().sort_index())

Unique source IDs: 1028

Videos per source ID:
6     1000
9        1
11       1
14       2
15       2
16       1
18       1
23       1
24       1
26       1
28       2
32       1
35       1
42       1
44       2
46       2
49       1
52       1
54       2
60       1
61       1
64       1
76       1
Name: count, dtype: int64


In [13]:
df[df["File Path"].str.startswith("DeepFakeDetection")][
    ["File Path", "Label"]
].head(20).to_string(index=False)

'                                                               File Path Label\nDeepFakeDetection/06_18__walking_down_street_outside_angry__MVXBDKGB.mp4  FAKE\n             DeepFakeDetection/03_07__talking_against_wall__PWXXULHR.mp4  FAKE\n                  DeepFakeDetection/20_25__exit_phone_room__M86GQTHK.mp4  FAKE\n             DeepFakeDetection/02_15__walk_down_hall_angry__TN2CWM3K.mp4  FAKE\n   DeepFakeDetection/16_05__walking_outside_cafe_disgusted__053HPS1G.mp4  FAKE\nDeepFakeDetection/14_21__walking_down_street_outside_angry__N0WM4GSV.mp4  FAKE\n     DeepFakeDetection/06_14__outside_talking_pan_laughing__8U9ULZDT.mp4  FAKE\n             DeepFakeDetection/03_26__talking_against_wall__WBSXBL82.mp4  FAKE\n                    DeepFakeDetection/01_27__hugging_happy__ZYCZ30C0.mp4  FAKE\n              DeepFakeDetection/15_03__talking_angry_couch__T5SHH2UL.mp4  FAKE\n                    DeepFakeDetection/14_18__hugging_happy__0YRBHIKG.mp4  FAKE\n              DeepFakeDetection/20_15__

In [14]:
print(
    df[df["File Path"].str.startswith("DeepFakeDetection")]["File Path"]
    .head(50)
    .tolist()
)

['DeepFakeDetection/06_18__walking_down_street_outside_angry__MVXBDKGB.mp4', 'DeepFakeDetection/03_07__talking_against_wall__PWXXULHR.mp4', 'DeepFakeDetection/20_25__exit_phone_room__M86GQTHK.mp4', 'DeepFakeDetection/02_15__walk_down_hall_angry__TN2CWM3K.mp4', 'DeepFakeDetection/16_05__walking_outside_cafe_disgusted__053HPS1G.mp4', 'DeepFakeDetection/14_21__walking_down_street_outside_angry__N0WM4GSV.mp4', 'DeepFakeDetection/06_14__outside_talking_pan_laughing__8U9ULZDT.mp4', 'DeepFakeDetection/03_26__talking_against_wall__WBSXBL82.mp4', 'DeepFakeDetection/01_27__hugging_happy__ZYCZ30C0.mp4', 'DeepFakeDetection/15_03__talking_angry_couch__T5SHH2UL.mp4', 'DeepFakeDetection/14_18__hugging_happy__0YRBHIKG.mp4', 'DeepFakeDetection/20_15__podium_speech_happy__99NHTFBM.mp4', 'DeepFakeDetection/02_06__podium_speech_happy__N8OSN8P6.mp4', 'DeepFakeDetection/12_13__walk_down_hall_angry__6G44TDN4.mp4', 'DeepFakeDetection/06_04__walking_outside_cafe_disgusted__ZK95PQDE.mp4', 'DeepFakeDetection/14_

In [17]:
import pandas as pd
import os

BASE = os.path.join(dataset_path, "FaceForensics++_C23")
CSV = os.path.join(BASE, "csv", "FF++_Metadata_Shuffled.csv")

df = pd.read_csv(CSV)

# Exclude DeepFakeDetection for the initial experiment
methods = [
    "original",
    "Deepfakes",
    "FaceSwap",
    "FaceShifter",
    "Face2Face",
    "NeuralTextures"
]

df6 = df[df["File Path"].str.split("/").str[0].isin(methods)].copy()

def get_source_id(path):
    filename = os.path.basename(path).replace(".mp4", "")

    if "_" not in filename:
        return int(filename)

    return int(filename.split("_")[0])

df6["source_id"] = df6["File Path"].apply(get_source_id)

print("Videos:", len(df6))
print("Unique source IDs:", df6["source_id"].nunique())
print("\nLabels:")
print(df6["Label"].value_counts())
print("\nVideos per source:")
print(df6.groupby("source_id").size().value_counts())

Videos: 6000
Unique source IDs: 1000

Labels:
Label
FAKE    5000
REAL    1000
Name: count, dtype: int64

Videos per source:
6    1000
Name: count, dtype: int64


In [16]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [18]:
from sklearn.model_selection import train_test_split

# One row per source video
sources = sorted(df6["source_id"].unique())

# 70% train, 30% temporary
train_sources, temp_sources = train_test_split(
    sources,
    test_size=0.30,
    random_state=42
)

# Split remaining 30% into 15% val + 15% test
val_sources, test_sources = train_test_split(
    temp_sources,
    test_size=0.50,
    random_state=42
)

train_sources = set(train_sources)
val_sources = set(val_sources)
test_sources = set(test_sources)

# Assign split
df6["split"] = df6["source_id"].apply(
    lambda x: "train" if x in train_sources
    else "val" if x in val_sources
    else "test"
)

print(df6["split"].value_counts())

print("\nSource counts:")
print("Train:", len(train_sources))
print("Val:", len(val_sources))
print("Test:", len(test_sources))

print("\nLabel distribution:")
print(pd.crosstab(df6["split"], df6["Label"]))

split
train    4200
val       900
test      900
Name: count, dtype: int64

Source counts:
Train: 700
Val: 150
Test: 150

Label distribution:
Label  FAKE  REAL
split            
test    750   150
train  3500   700
val     750   150


In [19]:
print(
    set(df6[df6["split"]=="train"]["source_id"]) &
    set(df6[df6["split"]=="test"]["source_id"])
)

set()


In [21]:
import os

PROJECT_DIR = "/content/drive/MyDrive/Multimedia_Authentication"

os.makedirs(f"{PROJECT_DIR}/metadata", exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/splits", exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/models", exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/results", exist_ok=True)

# Save the final 6000-video dataset with source IDs and splits
split_path = f"{PROJECT_DIR}/splits/ffpp_6000_split.csv"

df6.to_csv(split_path, index=False)

print("Saved:", split_path)
print("Rows:", len(df6))

Saved: /content/drive/MyDrive/Multimedia_Authentication/splits/ffpp_6000_split.csv
Rows: 6000
